# C-MET 完整全流程复现

这个笔记本不是 demo 版，而是完整复现入口：数据检查、预处理、特征抽取、训练、推理、评估都从这里开始。

你需要先把 MEAD 和 CREMA-D 数据放到 Google Drive。完整训练建议使用 A100，高 RAM，Drive 剩余空间最好 1TB 以上。

## 1. 检查 GPU 和挂载 Google Drive

完整复现不要用 CPU。先确认 GPU，再挂载 Drive 保存数据、checkpoint 和结果。

In [ ]:
!nvidia-smi
!python --version

from google.colab import drive
drive.mount('/content/drive')

## 2. 设置完整复现路径

如果你的 Drive 路径不同，只改下面这个单元。后面所有步骤都引用这些变量。

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/C-MET-full')
DATASET_ROOT = DRIVE_ROOT / 'dataset'
MEAD_FPS25 = DATASET_ROOT / 'MEAD' / 'FPS25'
CREMAD_FPS25 = DATASET_ROOT / 'CREMA_D' / 'FPS25'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
PRETRAINED_DIR = DRIVE_ROOT / 'pretrained_weights'
RESULT_DIR = DRIVE_ROOT / 'res'
TB_DIR = DRIVE_ROOT / 'tensorboard_runs'

for path in [DRIVE_ROOT, DATASET_ROOT, MEAD_FPS25, CREMAD_FPS25, CHECKPOINT_DIR, PRETRAINED_DIR, RESULT_DIR, TB_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('完整复现根目录:', DRIVE_ROOT)
print('MEAD 目标目录:', MEAD_FPS25)
print('CREMA-D 目标目录:', CREMAD_FPS25)

## 3. 克隆官方 C-MET 和复现工具仓库

官方仓库负责模型代码；本仓库负责 Colab 补丁、检查脚本和批处理脚本。

In [ ]:
%cd /content
!rm -rf C-MET cmet-repro-colab
!git clone https://github.com/ChanHyeok-Choi/C-MET.git
!git clone https://github.com/ChengyangHe-ux/cmet-repro-colab.git
!mkdir -p /content/C-MET/repro_tools
!cp /content/cmet-repro-colab/scripts/*.py /content/C-MET/repro_tools/
%cd /content/C-MET

## 4. 安装依赖

这里安装完整流程需要的核心依赖。超分、Gradio、baseline 环境后面单独装，避免一开始污染训练环境。

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg pkg-config

!python -m pip install --no-cache-dir \
  'setuptools<82' 'jedi>=0.16' \
  huggingface_hub==1.20.1 omegaconf==2.3.0 tqdm==4.67.3 \
  pandas==2.2.2 scipy==1.16.3 tensorboard \
  librosa==0.10.2.post1 'soundfile>=0.12.1' \
  imageio==2.37.3 imageio-ffmpeg==0.6.0 \
  opencv-python-headless==4.13.0.92 pillow==11.3.0 \
  funasr face_alignment==1.4.1 torchdiffeq==0.2.5 timm==1.0.9 ninja tyro

## 5. 给官方代码打 Colab 兼容补丁

补丁只处理运行兼容问题，不改模型结构。

In [ ]:
!python repro_tools/patch_cmet_colab_full.py --cmet-root /content/C-MET

## 6. 连接 Drive 数据、checkpoint 和结果目录

官方代码自带 `dataset/MEAD/train.txt` 和 `test.txt`，所以这里不替换整个 `dataset` 目录，只把大数据目录 `FPS25` 接到 Drive。checkpoint、结果和 TensorBoard 也接到 Drive，避免 Colab 断开后丢数据。

In [ ]:
import os
from pathlib import Path

%cd /content/C-MET

def safe_link(link_path, target_path):
    link_path = Path(link_path)
    target_path = Path(target_path)
    target_path.mkdir(parents=True, exist_ok=True)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    if link_path.exists():
        print('路径已存在，未覆盖:', link_path)
        return
    os.symlink(target_path, link_path)
    print(link_path, '->', target_path)

safe_link('/content/C-MET/dataset/MEAD/FPS25', MEAD_FPS25)
safe_link('/content/C-MET/dataset/CREMA_D/FPS25', CREMAD_FPS25)
safe_link('/content/C-MET/checkpoints', CHECKPOINT_DIR)
safe_link('/content/C-MET/pretrained_weights', PRETRAINED_DIR)
safe_link('/content/C-MET/res', RESULT_DIR)
safe_link('/content/C-MET/tensorboard_runs', TB_DIR)

## 7. 下载官方依赖权重

EDTalk 特征抽取和推理需要这些预训练权重。这里下载到 Drive 软链接目录，避免下次重新下载。

In [ ]:
from huggingface_hub import hf_hub_download

for filename in ['Audio2Lip.pt', 'EDTalk.pt', 'EDTalk-V.pt']:
    path = hf_hub_download(
        repo_id='coldhyuk/C-MET',
        filename=f'pretrained_weights/{filename}',
        local_dir='/content/C-MET',
    )
    print('已准备:', path)

## 8. 检查完整数据结构

如果这个单元失败，先修数据结构。不要跳过直接训练。

In [ ]:
!python repro_tools/validate_full_dataset.py \
  --cmet-root /content/C-MET \
  --mead-root /content/C-MET/dataset/MEAD/FPS25

## 9. 抽取 emotion2vec+large 音频特征

这一步会遍历 MEAD 音频并生成语音情绪特征。完整数据会跑很久。

In [ ]:
%cd /content/C-MET
!python extract_e2v+L.py --data_root ./dataset/MEAD/FPS25

## 10. 抽取 EDTalk 表情、姿态、唇形特征

这一步为训练 C-MET connector 准备视觉侧监督信号。完整 MEAD 会跑很久，建议 A100。

In [ ]:
%cd /content/C-MET
!python repro_tools/extract_edtalk_features.py \
  --cmet-root /content/C-MET \
  --data-root /content/C-MET/dataset/MEAD/FPS25 \
  --batch-size 100

## 11. 再次检查数据是否可训练

这次加 `--strict`。如果失败，说明特征还没抽齐。

In [ ]:
!python repro_tools/validate_full_dataset.py \
  --cmet-root /content/C-MET \
  --mead-root /content/C-MET/dataset/MEAD/FPS25 \
  --strict

## 12. 从头训练 C-MET connector

这是完整复现的核心训练。默认配置来自官方 `configs/train.yaml`。

In [ ]:
%cd /content/C-MET
!python train.py \
  --config ./configs/train.yaml \
  --mode mean \
  --num_feats 10 \
  --direction average \
  --ID same \
  --feature_type ED \
  --audio_encoder emotion2vec+large \
  --train bidir \
  --lambda_cnt 0.1 \
  --lambda_dir 0.05 \
  --balance focal_mse

## 13. 查看 TensorBoard

训练时看 loss 曲线，确认不是只启动了进程。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/C-MET/tensorboard_runs

## 14. 用自训练 checkpoint 批量推理

把 `TRAINED_CHECKPOINT` 改成你训练出来的 checkpoint 路径。

In [ ]:
TRAINED_CHECKPOINT = './checkpoints/把这里改成你的训练目录/xxx_checkpoint_step000000000.pth'

!python repro_tools/batch_inference.py \
  --cmet-root /content/C-MET \
  --checkpoint "{TRAINED_CHECKPOINT}" \
  --out-dir ./res/full_repro

## 15. 基础视频评估

这一步会导出 CSV，记录分辨率、FPS、时长、音频流等基础信息。论文级 SyncNet/ArcFace/FVD 指标后续继续补。

In [ ]:
!python repro_tools/evaluate_videos_basic.py \
  --video-dir ./res/full_repro \
  --out-csv ./res/full_repro/video_basic_metrics.csv

## 16. 消融实验入口

完整复现需要跑消融。不要覆盖默认实验，给每组保留独立日志和 checkpoint。

In [ ]:
# 示例 1：去掉 contrastive loss
# !python train.py --config ./configs/train.yaml --lambda_cnt 0 --lambda_dir 0.05

# 示例 2：去掉 direction loss
# !python train.py --config ./configs/train.yaml --lambda_cnt 0.1 --lambda_dir 0

# 示例 3：单向训练
# !python train.py --config ./configs/train.yaml --train unidir